# 📧 Détection de Spam par Machine Learning
**Barbara KENGNE — Licence 3 Informatique, Aix-Marseille Université**

Ce projet explore la détection automatique de spams à partir d'un dataset de SMS réels.  
L'objectif est de comparer plusieurs approches de classification et d'analyser leurs performances.

---

## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliothèques importées avec succès')

## 2. Chargement et exploration des données

In [ ]:
# Chargement du dataset
df = pd.read_csv('spam.csv', encoding='latin-1', usecols=[0, 1])
df.columns = ['label', 'message']

print(f'Dataset chargé : {df.shape[0]} messages, {df.shape[1]} colonnes')
df.head()

In [ ]:
# Distribution des classes
counts = df['label'].value_counts()
print(counts)
print(f"\nProportion de spam : {counts['spam'] / len(df) * 100:.2f}%")
print(f"Proportion de ham  : {counts['ham'] / len(df) * 100:.2f}%")

In [ ]:
# Visualisation de la distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Diagramme en barres
colors = ['#2ecc71', '#e74c3c']
counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Distribution des messages (Ham vs Spam)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Catégorie')
axes[0].set_ylabel('Nombre de messages')
axes[0].tick_params(rotation=0)
for i, v in enumerate(counts):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Camembert
axes[1].pie(counts, labels=['Ham (légitime)', 'Spam'], colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Répartition en pourcentage', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Analyse exploratoire (EDA)

In [ ]:
# Longueur des messages
df['longueur'] = df['message'].apply(len)

print('Longueur moyenne des messages :')
print(df.groupby('label')['longueur'].describe().round(2))

In [ ]:
# Distribution de la longueur par catégorie
fig, ax = plt.subplots(figsize=(10, 4))

df[df['label'] == 'ham']['longueur'].plot(kind='hist', bins=50, alpha=0.6,
                                           color='#2ecc71', label='Ham', ax=ax)
df[df['label'] == 'spam']['longueur'].plot(kind='hist', bins=50, alpha=0.6,
                                            color='#e74c3c', label='Spam', ax=ax)

ax.set_xlabel('Longueur du message (caractères)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution de la longueur des messages', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('longueur_messages.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Nuage de mots pour les spams
spam_text = ' '.join(df[df['label'] == 'spam']['message'])
ham_text  = ' '.join(df[df['label'] == 'ham']['message'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wc_spam = WordCloud(width=600, height=400, background_color='white',
                    colormap='Reds', max_words=80).generate(spam_text)
axes[0].imshow(wc_spam, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Mots fréquents dans les SPAMS', fontsize=13, fontweight='bold', color='#e74c3c')

wc_ham = WordCloud(width=600, height=400, background_color='white',
                   colormap='Greens', max_words=80).generate(ham_text)
axes[1].imshow(wc_ham, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Mots fréquents dans les HAM', fontsize=13, fontweight='bold', color='#2ecc71')

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Prétraitement des données

In [ ]:
# Encodage des labels : ham = 0, spam = 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Séparation features / cible
X = df['message']
y = df['label_num']

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Taille train : {len(X_train)} messages')
print(f'Taille test  : {len(X_test)} messages')

In [ ]:
# Vectorisation TF-IDF
# TF-IDF mesure l'importance d'un mot dans un message
# relativement à l'ensemble du corpus
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)  # unigrammes + bigrammes
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Matrice TF-IDF : {X_train_tfidf.shape[0]} messages × {X_train_tfidf.shape[1]} features')

## 5. Entraînement et comparaison de 3 modèles

In [ ]:
# Définition des modèles
modeles = {
    'Naive Bayes (MultinomialNB)': MultinomialNB(),
    'Régression Logistique':       LogisticRegression(max_iter=1000),
    'SVM Linéaire (LinearSVC)':    LinearSVC(max_iter=2000)
}

resultats = []

for nom, modele in modeles.items():
    modele.fit(X_train_tfidf, y_train)
    y_pred = modele.predict(X_test_tfidf)

    resultats.append({
        'Modèle':    nom,
        'Accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score':  round(f1_score(y_test, y_pred) * 100, 2)
    })

df_resultats = pd.DataFrame(resultats)
print('\n📊 Comparaison des modèles :\n')
print(df_resultats.to_string(index=False))

In [ ]:
# Visualisation des performances
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(df_resultats))
width = 0.2
metriques = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors_m = ['#3498db', '#9b59b6', '#e67e22', '#1abc9c']

for i, (metrique, color) in enumerate(zip(metriques, colors_m)):
    bars = ax.bar(x + i * width, df_resultats[metrique], width,
                  label=metrique, color=color, alpha=0.85, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.3,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(df_resultats['Modèle'], fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_ylim(85, 102)
ax.set_title('Comparaison des performances — 3 modèles de détection de spam', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Analyse du meilleur modèle

In [ ]:
# Sélection du meilleur modèle selon le F1-Score
meilleur_nom = df_resultats.loc[df_resultats['F1-Score'].idxmax(), 'Modèle']
meilleur_modele = modeles[meilleur_nom]
print(f'✅ Meilleur modèle : {meilleur_nom}')

# Rapport de classification complet
y_pred_best = meilleur_modele.predict(X_test_tfidf)
print('\n📋 Rapport de classification :\n')
print(classification_report(y_test, y_pred_best, target_names=['Ham', 'Spam']))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'],
            yticklabels=['Ham', 'Spam'],
            linewidths=0.5, ax=ax)

ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Réalité', fontsize=12)
ax.set_title(f'Matrice de confusion — {meilleur_nom}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Vrais positifs (spam détecté)     : {tp}')
print(f'Faux positifs (ham classé spam)   : {fp}')
print(f'Faux négatifs (spam non détecté)  : {fn}')
print(f'Vrais négatifs (ham bien classé)  : {tn}')

In [ ]:
# Validation croisée (5 folds) pour confirmer la robustesse
print('🔄 Validation croisée (5 folds) :\n')
for nom, modele in modeles.items():
    X_all_tfidf = tfidf.transform(X)
    scores = cross_val_score(modele, X_all_tfidf, y, cv=5, scoring='f1')
    print(f'{nom:<35} F1 moyen = {scores.mean()*100:.2f}% (± {scores.std()*100:.2f}%)')

## 7. Tester le modèle en direct

In [ ]:
def predire(message):
    """Prédit si un message est spam ou ham."""
    vecteur = tfidf.transform([message])
    prediction = meilleur_modele.predict(vecteur)[0]
    label = '🚫 SPAM' if prediction == 1 else '✅ HAM (légitime)'
    print(f'Message : "{message}"')
    print(f'Résultat : {label}\n')

# Tests
predire("Congratulations! You've won a free iPhone. Click here now!")
predire("Hey, are you free for lunch tomorrow?")
predire("URGENT: Your account has been compromised. Call us immediately!")
predire("Don't forget the meeting at 3pm today.")

## 8. Conclusion

Ce projet a permis de construire et comparer trois modèles de machine learning pour la détection de spam :

| Modèle | Points forts |
|--------|-------------|
| **Naive Bayes** | Rapide, interprétable, bon point de départ |
| **Régression Logistique** | Équilibre précision/rappel, stable |
| **SVM Linéaire** | Souvent le plus performant sur données textuelles |

**Ce que j'ai appris :**
- L'importance de la vectorisation TF-IDF avec bigrammes pour capturer les expressions
- Le compromis précision/rappel : minimiser les faux négatifs est crucial en détection de spam
- La validation croisée pour évaluer la robustesse réelle d'un modèle

**Pistes d'amélioration futures :**
- Utiliser des modèles de deep learning (LSTM, BERT) pour de meilleures représentations sémantiques
- Tester sur d'autres langues (français, espagnol)
- Déployer le modèle via une API Flask ou FastAPI